# Day 67：面试专题(一)——技术八股：Transformer / RAG / Agent / Python 工程高频笔试与手撕代码（Jupyter Notebook 实战手册）

> **所属阶段**：第 6 阶段 · 第六阶段：毕业设计与就业冲刺 ｜ **第 10 周**：毕业答辩、简历作品集与面试冲刺结业  
> **GitLab 工单**：`STARMIND-067` ｜ **开发分支**：`feat/day67-interview-coding-fundamentals`  
> **配套 HTML5 课件**：`courseware/day67/index.html` ｜ **独立版本快照**：`versions/day67/`

---

## 🎙️ 老陈开场旁白

早啊各位！国内大厂（字节、阿里、腾讯、美团、Kimi/MiniMax 等）AI 应用工程师一面，90% 都会考两块硬菜：第一块是 **大模型与 RAG/Agent 核心原理八股**，比如'为什么 Attention 要除以 $\sqrt{d_k}$？'、'Top-K 和 Top-P 采样怎么手写？'、'KV Cache 为什么能加速 Decode？'；第二块是 **Python 后端工程高频手撕**，比如'手写一个带 TTL 过期的 LRU 缓存'、'手写一个支持突发流量的令牌桶限流器（Token Bucket）'。今天咱们不靠调包，全部用纯 Python 零依赖手撕一遍，并且用单元测试把边界条件全部卡死！

### 承上启下
- **昨日基线**：昨天我们用 STAR 法则打磨好了简历并生成了高转化率 GitHub 作品集，今天进入大厂一面最硬核的'技术八股 + 白板手撕代码'专项特训！
- **明日预告**：明天（Day 68）进入面试专题（二）：项目深挖防问穿与日活十万级企业知识库系统设计及容量估算。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day67'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-067）

AI 应用研发岗位一面普遍考察候选人对 Transformer 基础原理、解码采样策略、RAG/Agent 机制以及 Python 高并发后端基础设施的底层手写能力。

**今日迭代目标：**
- 纯 Python 手撕数值稳定的 Softmax、Scaled Dot-Product Attention（支持 Causal Mask）与 Top-K/Top-P 核采样。
- 纯 Python 手撕 O(1) 复杂度的 TTL-LRU 缓存（基于 `OrderedDict`）与令牌桶限流器（Token Bucket）。

**验收标准：**
- Attention 输出权重每行严格归一化和为 1，开启 Causal Mask 后上三角未来位置权重严格为 0。
- Top-P 核采样正确截断尾部低概率 token 并重新归一化。
- TTL-LRU 缓存正确执行过期淘汰与容量淘汰；令牌桶按速率补充并拦截超额突发流量。

---

## 📚 课堂核心技术笔记

### 1. 面试官追问：Temperature=0 和 Top-P 一起设有什么区别？

Temperature 越小分布越尖锐（趋近 Greedy 贪心解码，确定性高，适合代码/RAG 问答）；Temperature 越大分布越平坦（发散性强，适合创意写作）。工程上 RAG 与结构化 JSON 输出推荐 `temperature=0.0~0.2`。

> **⚠️ 老陈避坑**：手写 Softmax 如果忘了减 `max(logits)`，面试官输入 `[1000, 1001]` 直接把你的代码打成 `OverflowError`。

### 2. 面试官追问：为什么不用后台线程每秒往令牌桶塞令牌？

高并发多租户场景下如果有 10 万个租户，起后台定时任务维护令牌极度浪费 CPU。工业界（包括 Redis Lua 脚本实现）统一采用'惰性计算（Lazy Refill）'：每次请求到达时，用 `(now - last_ts) * rate` 动态算出应补充的令牌数并封顶到 `capacity`。

> **⚠️ 老陈避坑**：注意用单调时钟或可注入时钟函数 `clock_fn` 避免系统时间回拨（NTP 校时）导致算出负数令牌，同时方便单元测试无需 `time.sleep`。


## 💻 实操步骤 1：步骤一：纯 Python 手撕 Scaled Dot-Product Attention、因果掩码与 Top-P 采样

- **目标文件**：`versions/day67/src/day67_handwritten_transformer.py`
- **老陈代码旁白**：不依赖 NumPy/PyTorch，用纯 Python 清晰手写大厂一面最高频的三个大模型核心算法：数值稳定 Softmax、因果注意力与 Top-P 核采样。


In [ ]:
"""
Day 67 - 模块一：面试高频手撕——数值稳定 Softmax、Scaled Dot-Product Attention 与 Top-P 采样
"""
from __future__ import annotations

import math
from typing import Dict, List, Tuple


def stable_softmax(scores: List[float]) -> List[float]:
    """数值稳定 Softmax：减去最大值防止 exp 溢出；支持 -inf 掩码位"""
    finite = [s for s in scores if not math.isinf(s)]
    if not finite:
        return [0.0 for _ in scores]
    m = max(finite)
    exps = [0.0 if (math.isinf(s) and s < 0) else math.exp(s - m) for s in scores]
    denom = sum(exps)
    return [e / denom for e in exps]


def scaled_dot_product_attention(
    Q: List[List[float]],
    K: List[List[float]],
    V: List[List[float]],
    causal_mask: bool = True,
) -> Tuple[List[List[float]], List[List[float]]]:
    """
    纯 Python 实现 Attention(Q, K, V) = softmax(Q K^T / sqrt(d_k) + mask) V
    返回: (output_matrix, attention_weights)
    """
    seq_len = len(Q)
    d_k = len(Q[0])
    d_v = len(V[0])
    scale = math.sqrt(d_k)

    weights: List[List[float]] = []
    for i in range(seq_len):
        row_scores: List[float] = []
        for j in range(seq_len):
            if causal_mask and j > i:
                row_scores.append(float("-inf"))
            else:
                dot = sum(Q[i][k] * K[j][k] for k in range(d_k))
                row_scores.append(dot / scale)
        weights.append(stable_softmax(row_scores))

    output: List[List[float]] = []
    for i in range(seq_len):
        vec = [0.0] * d_v
        for j in range(seq_len):
            w = weights[i][j]
            for d in range(d_v):
                vec[d] += w * V[j][d]
        output.append([round(x, 6) for x in vec])
    return output, weights


def top_p_filter(logits: Dict[str, float], top_p: float = 0.9, temperature: float = 1.0) -> Dict[str, float]:
    """Top-P (Nucleus) 核采样概率截断与重归一化"""
    if temperature <= 0:
        best_tok = max(logits.items(), key=lambda x: x[1])[0]
        return {best_tok: 1.0}

    tokens = list(logits.keys())
    probs = stable_softmax([logits[t] / temperature for t in tokens])
    ranked = sorted(zip(tokens, probs), key=lambda x: x[1], reverse=True)

    kept: List[Tuple[str, float]] = []
    cum = 0.0
    for tok, p in ranked:
        kept.append((tok, p))
        cum += p
        if cum >= top_p:
            break

    total = sum(p for _, p in kept)
    return {tok: round(p / total, 6) for tok, p in kept}


if __name__ == "__main__":
    Q = [[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]
    K = [[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]
    V = [[10.0, 0.0], [0.0, 20.0], [5.0, 5.0]]
    out, attn = scaled_dot_product_attention(Q, K, V, causal_mask=True)
    print("因果注意力权重行0:", attn[0])
    print("注意力输出:", out)
    print("Top-P=0.8 截断结果:", top_p_filter({"星": 3.0, "辰": 2.2, "智": 0.5, "脑": -1.0}, top_p=0.8))


### 🔍 步骤 1 代码深度解析

完整覆盖面试官必查的三大细节：`max(finite)` 防 `exp` 上溢、`j > i` 置 `-inf` 保证自回归因果性、`temperature <= 0` 退化为贪心解码。

**预期运行输出参考：**
```text
因果注意力权重行0: [1.0, 0.0, 0.0]
注意力输出: [[10.0, 0.0], ...]
Top-P=0.8 截断结果: {'星': ..., '辰': ...}
```


## 💻 实操步骤 2：步骤二：纯 Python 手撕令牌桶限流器与 O(1) TTL-LRU 缓存

- **目标文件**：`versions/day67/src/day67_engineering_interview.py`
- **老陈代码旁白**：手写后端一面最高频的两道工程代码题：支持时钟注入的惰性令牌桶限流器，以及基于 `OrderedDict` 的 O(1) 带过期时间 LRU 缓存。


In [ ]:
"""
Day 67 - 模块二：面试高频手撕——令牌桶限流器 (Token Bucket) 与 O(1) TTL-LRU 缓存
"""
from __future__ import annotations

import time
from collections import OrderedDict
from typing import Any, Callable, Optional, Tuple


class TokenBucketLimiter:
    """惰性计算令牌桶限流器：O(1) 时间复杂度，无需后台定时线程"""

    def __init__(
        self,
        capacity: float,
        refill_rate: float,
        clock_fn: Callable[[], float] = time.monotonic,
    ) -> None:
        self.capacity = float(capacity)
        self.refill_rate = float(refill_rate)  # tokens per second
        self._clock = clock_fn
        self._tokens = float(capacity)
        self._last_ts = self._clock()

    def allow(self, cost: float = 1.0) -> bool:
        now = self._clock()
        elapsed = max(0.0, now - self._last_ts)
        self._tokens = min(self.capacity, self._tokens + elapsed * self.refill_rate)
        self._last_ts = now
        if self._tokens >= cost:
            self._tokens -= cost
            return True
        return False


class TTLLRUCache:
    """O(1) 带 TTL 过期的 LRU 缓存（OrderedDict 实现）"""

    def __init__(
        self,
        maxsize: int = 128,
        ttl_seconds: float = 60.0,
        clock_fn: Callable[[], float] = time.monotonic,
    ) -> None:
        self.maxsize = max(1, maxsize)
        self.ttl_seconds = ttl_seconds
        self._clock = clock_fn
        self._store: OrderedDict[str, Tuple[Any, float]] = OrderedDict()

    def get(self, key: str) -> Optional[Any]:
        if key not in self._store:
            return None
        val, expire_at = self._store[key]
        if self._clock() >= expire_at:
            del self._store[key]
            return None
        self._store.move_to_end(key)
        return val

    def set(self, key: str, value: Any, ttl_seconds: Optional[float] = None) -> None:
        ttl = self.ttl_seconds if ttl_seconds is None else ttl_seconds
        expire_at = self._clock() + ttl
        if key in self._store:
            self._store.move_to_end(key)
        self._store[key] = (value, expire_at)
        while len(self._store) > self.maxsize:
            self._store.popitem(last=False)


if __name__ == "__main__":
    fake_now = [0.0]
    tb = TokenBucketLimiter(capacity=2, refill_rate=1.0, clock_fn=lambda: fake_now[0])
    print("突发3次请求:", [tb.allow(), tb.allow(), tb.allow()])
    fake_now[0] += 1.0
    print("1秒后补充1个令牌:", tb.allow())

    cache = TTLLRUCache(maxsize=2, ttl_seconds=10.0, clock_fn=lambda: fake_now[0])
    cache.set("k1", "v1")
    cache.set("k2", "v2")
    _ = cache.get("k1")  # k1 变热
    cache.set("k3", "v3")  # 淘汰最久未用的 k2
    print("LRU 淘汰验证 (k1, k2, k3):", cache.get("k1"), cache.get("k2"), cache.get("k3"))


### 🔍 步骤 2 代码深度解析

通过依赖注入 `clock_fn`，既保证生产默认使用不受 NTP 回拨影响的 `time.monotonic`，又让单元测试毫秒级精准验证时间推进与过期逻辑。

**预期运行输出参考：**
```text
突发3次请求: [True, True, False]
1秒后补充1个令牌: True
LRU 淘汰验证 (k1, k2, k3): v1 None v3
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day67/tests/test_day67.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 67 单元测试：手撕 Attention/Top-P 与令牌桶/TTL-LRU"""
import math
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from src.day67_handwritten_transformer import scaled_dot_product_attention, stable_softmax, top_p_filter
from src.day67_engineering_interview import TTLLRUCache, TokenBucketLimiter


def test_stable_softmax_handles_large_logits():
    probs = stable_softmax([1000.0, 1001.0, 1000.0])
    assert math.isclose(sum(probs), 1.0, rel_tol=1e-6)
    assert probs[1] > probs[0]


def test_causal_attention_blocks_future_tokens():
    Q = [[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]
    out, w = scaled_dot_product_attention(Q, Q, Q, causal_mask=True)
    assert w[0][1] == 0.0 and w[0][2] == 0.0
    assert w[1][2] == 0.0
    for row in w:
        assert math.isclose(sum(row), 1.0, rel_tol=1e-6)
    assert len(out) == 3


def test_top_p_truncates_tail_and_renormalizes():
    filtered = top_p_filter({"A": 4.0, "B": 1.0, "C": -5.0}, top_p=0.95)
    assert "A" in filtered and "C" not in filtered
    assert math.isclose(sum(filtered.values()), 1.0, rel_tol=1e-5)
    assert top_p_filter({"A": 1.0, "B": 2.0}, temperature=0.0) == {"B": 1.0}


def test_token_bucket_and_ttl_lru_cache():
    now = [100.0]
    tb = TokenBucketLimiter(capacity=2, refill_rate=0.5, clock_fn=lambda: now[0])
    assert tb.allow() is True and tb.allow() is True and tb.allow() is False
    now[0] += 2.0
    assert tb.allow() is True

    cache = TTLLRUCache(maxsize=2, ttl_seconds=5.0, clock_fn=lambda: now[0])
    cache.set("a", 1)
    cache.set("b", 2)
    assert cache.get("a") == 1
    cache.set("c", 3)
    assert cache.get("b") is None and cache.get("c") == 3
    now[0] += 6.0
    assert cache.get("a") is None and cache.get("c") is None

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：基础巩固：手写 Top-K 采样过滤器

**题目要求：**
实现 `top_k_filter(logits: dict[str, float], k: int) -> dict[str, float]`，仅保留 logit 最高的 K 个 token 并返回 Softmax 归一化概率。

**思路提示：** 先按 logit 降序切片 `[:k]`，再调用 `stable_softmax`。


In [ ]:
from src.day67_handwritten_transformer import stable_softmax


def top_k_filter(logits: dict[str, float], k: int) -> dict[str, float]:
    top = sorted(logits.items(), key=lambda x: x[1], reverse=True)[:k]
    probs = stable_softmax([v for _, v in top])
    return {tok: round(p, 6) for (tok, _), p in zip(top, probs)}


res = top_k_filter({"a": 3.0, "b": 2.0, "c": 1.0, "d": -2.0}, k=2)
assert set(res.keys()) == {"a", "b"} and abs(sum(res.values()) - 1.0) < 1e-5
print("Top-K 过滤通过:", res)


**答案解析与评分标准：**

工业推理引擎（vLLM/SGLang）通常先做 Top-K 粗截断，再做 Top-P 动态核采样。


### 作业 2（企业实战）：企业实战：手写带指数退避（Exponential Backoff）的异步重试装饰器

**题目要求：**
编写 `async_retry(fn, max_retries=3, base_delay=0.01)`，遇到异常时按 `base_delay * (2 ** attempt)` 退避重试，成功即返回。

**思路提示：** 用 `for attempt in range(max_retries)` 捕获异常并 `await asyncio.sleep(...)`。


In [ ]:
import asyncio


async def async_retry(fn, max_retries: int = 3, base_delay: float = 0.005):
    last_err = None
    for attempt in range(max_retries):
        try:
            return await fn()
        except Exception as exc:
            last_err = exc
            if attempt < max_retries - 1:
                await asyncio.sleep(base_delay * (2 ** attempt))
    raise last_err


calls = {"n": 0}


async def flaky():
    calls["n"] += 1
    if calls["n"] < 3:
        raise RuntimeError("429 rate limit")
    return "OK"


assert asyncio.run(async_retry(flaky)) == "OK" and calls["n"] == 3
print("指数退避重试通过")


**答案解析与评分标准：**

调用外部大模型 API 遇到 429/502 时，指数退避 + 随机抖动（Jitter）是防止重试风暴的标准方案。


### 作业 3（架构进阶）：架构进阶：计算 7B 模型单请求 KV Cache 显存占用（MB）

**题目要求：**
编写 `calc_kv_cache_mb(layers=28, kv_heads=4, head_dim=128, seq_len=4096, dtype_bytes=2)` 计算 GQA 架构下单个长上下文请求的 KV Cache 占用（MB）。

**思路提示：** 公式：`2 (K和V) * layers * kv_heads * head_dim * seq_len * dtype_bytes / (1024 ** 2)`。


In [ ]:
def calc_kv_cache_mb(layers: int = 28, kv_heads: int = 4, head_dim: int = 128, seq_len: int = 4096, dtype_bytes: int = 2) -> float:
    total_bytes = 2 * layers * kv_heads * head_dim * seq_len * dtype_bytes
    return round(total_bytes / (1024 ** 2), 2)


mb = calc_kv_cache_mb()
assert mb == 224.0
print(f"Qwen2.5-7B (GQA 4 KV-heads) 4K 上下文单请求 KV Cache = {mb} MB")


**答案解析与评分标准：**

懂 GQA（Grouped-Query Attention）为何大幅压缩 KV Cache 显存，是面试推理部署岗位的高频加分点。


---

## 🎯 老陈收尾复盘

畅快！今天手撕的 Scaled Dot-Product Attention、因果掩码、Top-P 核采样、令牌桶限流器与 TTL-LRU 缓存，全是大厂面试真题里的'常青树'。考前把这几个模块闭眼写两遍，一面手撕稳如泰山！明天咱们攻克二面项目深挖与系统设计！
